In [0]:
%pip install -U mlflow scikit-learn
dbutils.library.restartPython()

In [0]:
import mlflow

dbutils.widgets.text("catalog", "ieee_cis")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

MODEL_NAME = f"{catalog}.ml.fraud_model"
user = spark.sql("SELECT current_user()").first()[0]

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{user}/fraud-detection")
mlflow.sklearn.autolog(log_models=False)   # auto-log params; we register the model ourselves

In [0]:
CAT_COLS = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
            "M4", "M6", "DeviceType"]
NUM_COLS = ["TransactionAmt", "card1", "card2", "card3", "card5",
            "addr1", "addr2", "dist1",
            *[f"C{i}" for i in range(1, 15)],
            "D1", "D4", "D10", "D15", "card_txn_24h"]
FEATURES = CAT_COLS + NUM_COLS

In [0]:
df = (spark.table("silver.train")
      .select("TransactionID", "TransactionDT", "isFraud", *FEATURES)
      .toPandas())
print(df.shape)

In [0]:
print(df)

In [0]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2

# toPandas() doesn't guarantee row order, so sort by time first.
# TransactionID breaks ties, so the order is identical on every run.
df = df.sort_values(["TransactionID","TransactionDT"]).reset_index(drop=True)

train, valid = train_test_split(df, test_size=TEST_SIZE, shuffle=False) #  shuffle=False keeps time order
cutoff  = train["TransactionDT"].max() # logged for 03_evaluate


X_train, y_train = train[FEATURES], train["isFraud"]
X_valid, y_valid = valid[FEATURES], valid["isFraud"]


assert len(train) + len(valid) == len(df)
print(f"train={len(train):,}  valid={len(valid):,}  "
      f"fraud train={y_train.mean():.4f}  valid={y_valid.mean():.4f}")


In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

encode = ColumnTransformer(transformers=[
        (
            "encoder", 
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,encoded_missing_value=-1), 
            CAT_COLS
        )
    ],
    remainder="passthrough"
)

clf = HistGradientBoostingClassifier(
    class_weight="balanced",
    categorical_features=list(range(len(CAT_COLS))),
    random_state=42,
)

model = Pipeline([("encode", encode),('clf',clf)])


In [0]:
%skip
model.fit(X_train.head(5000), y_train.head(5000))

In [0]:
from sklearn.metrics import average_precision_score, roc_auc_score

with mlflow.start_run(run_name="hgb_v1") as run:
    # 1. Record how this model was made (03_evaluate reads cutoff/test_size back)
    mlflow.log_params({
        "cutoff": cutoff,
        "test_size": TEST_SIZE,
        "split": "time_ordered",
        "n_features": len(FEATURES),
        "train_rows": len(train),
        "valid_rows": len(valid),
    })

    # 2. Train on the older 80%
    model.fit(X_train, y_train)

    # 3. Quick score on the newest 20% (the full evaluation happens in 03_evaluate)
    proba = model.predict_proba(X_valid)[:, 1]        # column 1 = probability of fraud
    val_auprc = average_precision_score(y_valid, proba)
    val_roc_auc = roc_auc_score(y_valid, proba)
    mlflow.log_metrics({"val_auprc": val_auprc, "val_roc_auc": val_roc_auc})

    # 4. Save the whole pipeline (encoder + model) and register it as a new version
    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        input_example=X_train.sample(100, random_state=0),   # lets MLflow infer the signature UC requires
        registered_model_name=MODEL_NAME,
    )

print(f"val_auprc   = {val_auprc:.4f}   (random guessing ≈ {y_valid.mean():.4f})")
print(f"val_roc_auc = {val_roc_auc:.4f}   (random guessing = 0.5)")
print(f"registered  = {MODEL_NAME} v{model_info.registered_model_version}")